In [ ]:
# Customer Churn — Baseline Notebook

**RabTech Academy internship | Task 2**

### Purpose
This notebook inspects the supplied customer-churn dataset and establishes simple, non-ML baselines before any advanced model is trained.

**Important limitation:** the dataset contains only 12 records. Any results here are exploratory and are not reliable evidence of real-world predictive performance. The dataset's original source, permission terms, target definition, and label time window have not been verified.

In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = "customer-churn-training.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape (rows, columns):", df.shape)
display(df.head())

In [ ]:
## 2. Inspect columns and data quality
We check column types, missing values, exact duplicate rows, and target counts. These checks describe the supplied file; they do not establish that the source data is accurate or representative.

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

print("\nMissing values per column:")
display(df.isna().sum().to_frame("missing_count"))

print("\nExact duplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
display(df["churned"].value_counts())

print("\nTarget proportions:")
display(df["churned"].value_counts(normalize=True))

In [ ]:
## 4. Baseline A — majority-class prediction
A majority-class baseline predicts the most common target value for every record. This is a simple reference point, not a useful personalized churn strategy.

The metrics below are calculated on the same 12 records used to describe the dataset. They are **not held-out test results** and must not be interpreted as evidence of generalization.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_true = df["churned"].astype(int)
majority_class = int(y_true.value_counts().idxmax())
y_pred_majority = np.full(len(df), majority_class)

def show_metrics(name, y_t, y_p):
    print(f"--- {name} ---")
    print("Accuracy  :", round(accuracy_score(y_t, y_p), 4))
    print("Precision :", round(precision_score(y_t, y_p, zero_division=0), 4))
    print("Recall    :", round(recall_score(y_t, y_p, zero_division=0), 4))
    print("F1-score  :", round(f1_score(y_t, y_p, zero_division=0), 4))
    cm = confusion_matrix(y_t, y_p)
    print("Confusion matrix:\n", cm)
    print("\n")

show_metrics("Majority-class baseline", y_true, y_pred_majority)

In [ ]:
## 5. Baseline B — simple inactivity rule
Example rule: flag a customer if `last_login_days >= 20`. The 20-day threshold is illustrative, not an established business rule. A business owner should approve the threshold based on context and costs.

This is a rule-based baseline, not a trained ML model. As above, the displayed metrics use the same tiny dataset to illustrate how evaluation works; they are not a valid estimate of future performance.

In [ ]:
INACTIVITY_THRESHOLD_DAYS = 20
y_pred_inactivity = (df["last_login_days"] >= INACTIVITY_THRESHOLD_DAYS).astype(int)

show_metrics("Inactivity rule baseline", y_true, y_pred_inactivity)

# Inspect predictions per customer
display(df[["customer_id", "last_login_days", "churned"]].assign(
    predicted_churn=y_pred_inactivity,
    correct_prediction=(y_pred_inactivity == y_true)
))

In [ ]:
## 6. False positives and false negatives
* **False positive (FP):** the rule flags a customer as likely to churn, but the recorded target is 0. This may cause unnecessary outreach or retention spending.
* **False negative (FN):** the rule does not flag a customer, but the recorded target is 1. This may mean a missed opportunity to retain the customer.
*

In [ ]:
cm = confusion_matrix(y_true, y_pred_inactivity)
if cm.shape == (2, 2):
    tn, fp, fn, tp = cm.ravel()
    print(f"True negatives (TN): {tn}")
    print(f"False positives (FP): {fp}")
    print(f"False negatives (FN): {fn}")
    print(f"True positives (TP): {tp}")
else:
    print("Confusion matrix shape is single-class:", cm)

In [ ]:
## 7. Leakage and responsible-use checks

Before modeling, verify:
* `churned` is the label and is not included in input features.
* Each feature was available at the intended prediction date.
* `churned` has a clear definition and a documented label window.
* The dataset source, permissions, and whether data is real or synthetic are confirmed.
* Customer identifiers are excluded from model inputs.
* Sensitive attributes and possible proxies are considered where appropriate.

The current CSV alone does not answer all these questions. Record unknown details as "unknown — verification required."

In [ ]:
## 8. Conclusion and limitations

This notebook establishes two simple reference baselines and documents basic quality checks. With only 12 records, the results are highly uncertain; splitting this dataset into train and test sets would leave too few records for meaningful evaluation. Do not claim that the baselines or any future model are production-ready.